<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/a/ac/World-airline-routemap-2009.png/1024px-World-airline-routemap-2009.png" width="700px;" alt="flightmap"/>

#   `lesson07`:  Working with *large* data

## Objectives

### Objectives

1. Identify **fast** methods for data analysis on *large* datasets
2. Use methods to reduce the amount of **memory** required  
3. Evaluate datasets in **chunks**

### Example Questions

1. With a large dataset, how can we analyze Series information in a dataframe in an efficient manner?
2. Is it faster to loop over rows or to access the underlying numpy array?
3. What are some mechanisms for reducing the memory footprint?
4. How can we convert column information to a limited number of types?
5. If the dataset is simply too large, how can we read it in piece by piece?

## Highlevel topics

- Performance
- Data thinning

## Synopsis

You have a large dataset (say 1GB or 10GB or 100GB) with both relevant and useless data.  You've built a an analysis tool, but it's too slow -- now what?

#### Your Task

Build an efficient implementation.

## Datasets

We'll mainly work fromthe following
    - http://stat-computing.org/dataexpo/2009/the-data.html
        - Here, the 2008 dataset will be around 1.5GB in memory
    - https://www.transtats.bts.gov/DL_SelectFields.asp
        - later we'll use this dataset

## Getting Started

There are many methods for increasing performance in your analysis tools, from accessing the underlying data storage directly (e.g. numpy) to reducing the memory size to working on a subset of the data.  A few relevant links related to this notebook are:
    - https://engineering.upside.com/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6
    - https://www.csvexplorer.com/blog/open-big-csv/
    - https://towardsdatascience.com/why-and-how-to-use-pandas-with-large-data-9594dda2ea4c
    
It's also important to point out a few areas that we will not cover.  When your analysis tools are computationally intensive it may be helpful outsource the computations to C or C++ or other compiled languages.  For this, tools like Cython (https://cython.org/) can be helpful.  In addition, you may have acccess to a cluster of machines for processing your data -- parallel packages such as dask (http://docs.dask.org/en/latest/dataframe.html) can be used to represent a *parallel* dataframe.  Finally, modern clusters or workstations may include accelerators such as GPUs -- packages such as pytorch and others have built-in support for using these devices: https://towardsdatascience.com/speed-up-your-algorithms-part-1-pytorch-56d8a4ae7051.

In [ ]:
import pandas as pd
import numpy as np
%matplotlib inline

### Start with a large dataset

In [ ]:
!wget https://bitbucket.org/lukeolson/cs199py-sp19b-data/raw/322d201ac7e2b641d64727698891b408269c4258/2008small.csv.bz2

In [ ]:
!bzip2 -d 2008small.csv.bz2

In [ ]:
!ls -l

## Read the data set

**!!!** Warning this may take a while

In [ ]:
df = pd.read_csv('2008small.csv')
df.fillna(0, inplace=True)

Let's take a look at the full dataset

In [ ]:
df.info()

**1.5GB** is large for our processing.  As a start, we'll take the first $n$ rows to work on some analysis tools.  Then revisit the full dataset later.

In [ ]:
dfn = df[:10000].copy()

In [ ]:
dfn.info()

## Attempt 1

Let's start with a basic function.  We're going to look at the average total delay time, a fairly simple function.  For this, we'll take the total of two columns `ArrDelay` and `DepDelay`, then divide by the total samples.

For this, let's take a look at building a function and applying it **per row** in our dataframe.  The `.iloc[i]` function will allow us to access row `i` of the table.

In [ ]:
%%timeit

total_time = 0
for i in range(len(dfn)):
    total_time += dfn.iloc[i]['ArrDelay'] + dfn.iloc[i]['DepDelay']
avg_time = total_time / len(dfn)

**this is only for a small number of rows!!** If we're taking a few seconds for small number of rows (1e4) then the time on the full dataset will grow rapidly.

## Attempt 2

Iterating directly over rows is **very** slow.

Luckily Pandas provides some other mechanisms.  For example the `.iterrows()` steps through the data in (index, series) pairs as follows

In [ ]:
%%timeit

total_time = 0
for index, row in dfn.iterrows():
    total_time += row['ArrDelay'] + row['DepDelay']
avg_time = total_time / len(dfn)

## Attempt 3

We can improve this further with the `.apply()` function which we've already used through the course.

Using `axis=1` will (oddly) apply to each row (meaning over columns) whereas `axis=0` will apply to each column.

In [ ]:
%%timeit
dfn['total_time'] = dfn.apply(lambda row: row['ArrDelay'] + row['DepDelay'], axis=1)
avg_time = dfn['total_time'].mean()

## Attempt 4

If we inspect each column, it's stored as a Series.  We can speed up the computations significantly if we can work on the Series as a whole.

In [ ]:
%%timeit
avg_time = (dfn['ArrDelay'] + dfn['DepDelay']).mean()

## Attempt 5: direct numpy access

Here we'll modify the last attempt to access the underlying numpy array with `.values`.  For numerical data this can very fast, using numpy's vectorization to efficintly perform computations:

In [ ]:
%%timeit
avg_time = (dfn['ArrDelay'].values + dfn['DepDelay'].values).mean()

Notice that we can now attempt a calculation on the full dataset (why?)

In [ ]:
%%timeit
avg_time = (df['ArrDelay'].values + df['DepDelay'].values).mean()

## Memory size

Let's go back to the full dataset.

7M rows, 29 columns for a total of 1GB+ of data.

In [ ]:
df.info()

But we're only using two columns for this analysis.  Let's grab only those two columns!

In [ ]:
df = df[['ArrDelay','DepDelay']].copy()

In [ ]:
df.info()

We're really *just* storing the 7009728 numbers (x2):

In [ ]:
len(df) * 8 * 2 / 1024 / 1024

In [ ]:
df['total_time'] = df['ArrDelay'] + df['DepDelay']
df['total_time'].mean()

In [ ]:
%%timeit
df['total_time'] = df['ArrDelay'] + df['DepDelay']
df['total_time'].mean()

In [ ]:
%%timeit
avg_time = (df['ArrDelay'] + df['DepDelay']).mean()

Above we simply extracted part of the data frame after reading it
in.  With Pandas, we can actually specifying which columns we
want when reading in the data to prevent ever having to store the
full dataframe in memory.

In [ ]:
df_small = pd.read_csv('2008.csv',
                       usecols=['ArrDelay','DepDelay'])
df_small.info()

## Precision and accuracy

The two columns we're looking at track the arrival and departure delays.  I think we could agree that these are not accurate to a 1/10 of a second!

One thing we can do is simply store these with less information.  A 16bit floating point number (versus the standard 64bit double floating point number) for example:

In [ ]:
df = df.astype('float16')

In [ ]:
df.info()

In [ ]:
len(df) * 2 * 3 / 1024 / 1024

In [ ]:
df

In [ ]:
df['total_time'] = df['ArrDelay'] + df['DepDelay']
np.mean(df['total_time'].values)

For the above, we changed the type to a lower precision after reading
it in, but before doin the analysis to make the analysis faster.
We can actually specify this when reading the data in as well to
prevent using the additional memory in the first place.

In [ ]:
df_small = pd.read_csv('2008.csv',
                       usecols=['ArrDelay','DepDelay'],
                       dtype={'ArrDelay':'float16', 'DepDelay':'float16'})
df_small.info()

## Chunks of data

Pandas also allows us to represent the data in chunks.  Let's take a look

In [ ]:
df_chunk = pd.read_csv('2008.csv', chunksize=100000)

In [ ]:
type(df_chunk)

What is this?  We need to *iterate* over `df_chunk`

In [ ]:
for chunk in df_chunk:
    print(type(chunk), len(chunk))

### Put it together

Let's iterate over each chunk and calculate the sums.  Then compute the total average.

In [ ]:
df_chunk = pd.read_csv('2008.csv', chunksize=100000)

In [ ]:
total_time = 0
total_size = 0

# Each chunk is in df format
for chunk in df_chunk:
    chunk.fillna(0, inplace=True)
    #print(type(chunk))
    total_time += (chunk['ArrDelay'].values + chunk['DepDelay'].values).sum()
    total_size += len(chunk)
avg_time = total_time / total_size
print("average delay: ", avg_time)

## Putting everything together
That still took a little bit of time, so to make it faster we
could combine the above technique with using a lower precision
and only reading in the columns we need.  Note that because
the sum will overflow a float16 value, we have to pass a higher
dtype to the sum command.

In [ ]:
df_chunk = pd.read_csv('2008.csv', chunksize=100000,
                       usecols=['ArrDelay','DepDelay'],
                       dtype={'ArrDelay':'float16', 'DepDelay':'float16'})
total_time = 0
total_size = 0

# Each chunk is in df format
for chunk in df_chunk:
    chunk.fillna(0, inplace=True)
    #print(type(chunk))
    total_time += (chunk['DepDelay'].values + chunk['ArrDelay'].values).sum(dtype=np.float64)
    total_size += len(chunk)
avg_time = total_time / total_size
print("average delay: ", avg_time)

## Your Turn
Your task is to perform a similar analysis on 2019 data.
The snippet below gets the data file.
This dataset has a lot of columns, but you'll only need to use
'DepDelay', 'ArrDelay', 'AirTime'.  Your task is to compute
for each flight the delay as a percentage of airtime and average
these percentage delays, i.e.
$$\mathtt{AvgPctDelay} = \frac{1}{n}\sum_{i=1}^{n} \frac{\mathtt{DepDelay} + \mathtt{ArrDelay}}{\mathtt{AirTime}}$$
where $n$ is the number of flights.

To do this efficiently, use the same approaches we took above.
Specifics:
* Read in the data with only those columns needed
* Use the float16 datatype for each column
* Read in the data by chunks
* If you call any sum command, pass `dtype=np.float64` as we did above to prevent an overflow in the sum.  
* Drop any rows with NaN values (using `dropna()`)

You'll submit your final answer $\mathtt{AvgPctDelay}$.

In [ ]:
!wget https://bitbucket.org/lukeolson/cs199py-sp19b-data/raw/322d201ac7e2b641d64727698891b408269c4258/2019.csv.bz2
!bzip2 -d 2019.csv.bz2
!ls -l

In [ ]:
# ANSWER - Remove this before adding to Azure
df19_chunk = pd.read_csv('2019.csv',
                         chunksize=10000,
                         usecols=['DepDelay', 'ArrDelay', 'AirTime'],
                         dtype={'DepDelay':'float16', 'ArrDelay':'float16','AirTime':'float16'})
total_pct = 0
n_flights = 0
for chunk in df19_chunk:
    chunk = chunk.dropna()
    total_pct += ((chunk['ArrDelay'].values + chunk['DepDelay'].values)/chunk['AirTime'].values).sum(dtype=np.float64)
    n_flights += len(chunk)
avg_pct = total_pct/n_flights
print(avg_pct)